# 04 — 研究问题整合分析

本 notebook 以题项均值、高分段比例、多选频数、Spearman 相关、Mann--Whitney 组间比较和访谈主题进行研究问题层面的探索性整合。

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
pd.set_option("display.max_colwidth", 120)

In [2]:
from src.analysis import (
    frequency_table,
    mann_whitney_group_table,
    slider_bin_agreement,
    spearman_correlation_matrix,
    spearman_correlation_table,
)
from src.data_loader import load_participant_survey, parse_survey_export
from src.participant_analysis import build_participant_scores
from src.preprocessing import attach_slider_constructs, construct_mean_summary

parsed = parse_survey_export(ROOT / "data" / "问卷原始数据.xlsx")
questions, options = parsed["questions"], parsed["options"]
sliders = attach_slider_constructs(parsed["sliders"])
agreement = slider_bin_agreement(options).merge(
    sliders[["q_num", "short_label", "construct", "mean"]],
    on="q_num",
    how="left",
)
constructs = construct_mean_summary(parsed["sliders"])
participants = load_participant_survey(ROOT / "data" / "processed" / "问卷数据_文本版.xlsx")
participant_scores = build_participant_scores(participants)

corr_variables = {
    "空间压迫/训练焦虑": "spatial_pressure",
    "社交媒体审美内化": "media_internalization",
    "训练自我效能": "training_self_efficacy",
    "干预偏好指数": "intervention_preference",
    "自由重量区回避频率": "avoidance_frequency_score",
}

## RQ1: 空间压迫与自由重量区回避

In [3]:
display(constructs[constructs["construct"].eq("空间压迫/训练焦虑")])
display(agreement[agreement["construct"].eq("空间压迫/训练焦虑")][["q_num", "short_label", "mean", "high_pct"]])
display(frequency_table(options, 16))
display(frequency_table(options, 17))
display(participant_scores[["spatial_pressure", "avoidance_frequency_score"]].corr(method="spearman"))

,construct,items,item_count,mean,min_item_mean,max_item_mean,valid_n
2,空间压迫/训练焦虑,"Q10, Q11, Q12, Q13, Q14, Q15",6,3.716667,3.19,4.24,32.0


,q_num,short_label,mean,high_pct
0,10,自由重量区不易接近,3.19,40.625
1,11,男性较多时不自在,4.24,81.250
2,12,镜子/公开可见增强被评价感,3.57,53.125
3,13,担心动作不标准被评价,3.81,65.625
4,14,自由重量区男性主导感,3.61,62.500
5,15,拥挤感降低停留意愿,3.88,75.000


,option,count,pct,count_status
68,从未,5.0,15.63,reported
69,很少,5.0,15.63,reported
70,有时,10.0,31.25,reported
71,经常,6.0,18.75,reported
72,总是,6.0,18.75,inferred_from_valid_n


,option,count,pct,count_status
73,男性太多,19.0,59.38,reported
74,害怕被注视,NaN,NaN,missing
75,害怕动作出错,9.0,28.13,reported
76,不会使用器械,19.0,59.38,reported
77,空间太拥挤,NaN,NaN,missing
78,布局/氛围太有压迫感,8.0,25.00,reported
79,不想练出明显肌肉,0.0,0.00,reported
80,更喜欢有氧训练,NaN,NaN,missing
81,没有同伴,11.0,34.38,reported
82,不知道从哪里开始,16.0,50.00,reported


,spatial_pressure,avoidance_frequency_score
spatial_pressure,1.000000,0.651777
avoidance_frequency_score,0.651777,1.000000


## 新增实验 1：主要变量 Spearman 相关

这里使用逐名文本版答卷矩阵。由于 $N=32$，结果只作为探索性双变量证据，不作为因果或预测模型。

In [4]:
correlations = spearman_correlation_table(participant_scores, corr_variables)
display(correlations.assign(
    spearman_rho=lambda d: d["spearman_rho"].round(3),
    p=lambda d: d["p"].round(4),
))

,x,y,n,spearman_rho,p
0,空间压迫/训练焦虑,社交媒体审美内化,32,0.405,0.0216
1,空间压迫/训练焦虑,训练自我效能,32,-0.021,0.9113
2,空间压迫/训练焦虑,干预偏好指数,32,0.671,0.0000
3,空间压迫/训练焦虑,自由重量区回避频率,32,0.652,0.0001
4,社交媒体审美内化,训练自我效能,32,0.145,0.4275
5,社交媒体审美内化,干预偏好指数,32,0.386,0.0290
6,社交媒体审美内化,自由重量区回避频率,32,0.310,0.0843
7,训练自我效能,干预偏好指数,32,0.265,0.1434
8,训练自我效能,自由重量区回避频率,32,-0.107,0.5617
9,干预偏好指数,自由重量区回避频率,32,0.455,0.0089


## 新增实验 2：相关矩阵

In [5]:
corr_matrix = spearman_correlation_matrix(participant_scores, corr_variables)
display(corr_matrix.round(3))

,空间压迫/训练焦虑,社交媒体审美内化,训练自我效能,干预偏好指数,自由重量区回避频率
空间压迫/训练焦虑,1.000,0.405,-0.021,0.671,0.652
社交媒体审美内化,0.405,1.000,0.145,0.386,0.310
训练自我效能,-0.021,0.145,1.000,0.265,-0.107
干预偏好指数,0.671,0.386,0.265,1.000,0.455
自由重量区回避频率,0.652,0.310,-0.107,0.455,1.000


## RQ2: 社交媒体审美内化与训练偏好

In [6]:
display(constructs[constructs["construct"].eq("社交媒体审美内化")])
display(agreement[agreement["construct"].eq("社交媒体审美内化")][["q_num", "short_label", "mean", "high_pct", "low_pct"]])
display(frequency_table(options, 9))

,construct,items,item_count,mean,min_item_mean,max_item_mean,valid_n
1,社交媒体审美内化,"Q21, Q22, Q23",3,2.66,1.85,3.4,32.0


,q_num,short_label,mean,high_pct,low_pct
6,21,审美内容影响运动选择,3.40,59.375,21.875
7,22,担心力量训练不够纤细,1.85,15.625,81.250
8,23,内化社交媒体女性身材标准,2.73,31.250,43.750


,option,count,pct,count_status
30,减脂/变瘦,18.0,56.25,reported
31,塑形/线条,11.0,34.38,reported
32,增强体能,11.0,34.38,reported
33,提高力量,6.0,18.75,reported
34,保持健康,8.0,25.00,reported
35,缓解压力,4.0,12.50,reported
36,社交需求,0.0,0.00,reported
37,其他 [详情],2.0,6.25,reported


## RQ3: 干预偏好与行动建议

In [7]:
display(constructs[constructs["construct"].eq("干预偏好指数")])
display(agreement[agreement["construct"].eq("干预偏好指数")][["q_num", "short_label", "mean", "high_pct"]])
display(frequency_table(options, 18))

,construct,items,item_count,mean,min_item_mean,max_item_mean,valid_n
0,干预偏好指数,"Q25, Q26, Q27",3,3.976667,3.39,4.5,32.0


,q_num,short_label,mean,high_pct
10,25,减少刻板印象会提升参与意愿,3.39,50.000
11,26,半私密分区会提升使用意愿,4.04,81.250
12,27,女性初学者workshop会提升尝试意愿,4.50,96.875


,option,count,pct,count_status
84,选择人少的时间去,15.0,46.88,reported
85,和朋友一起去,NaN,NaN,missing
86,只停留在有氧/固定器械区,12.0,37.50,reported
87,缩短在健身房停留的时间,2.0,6.25,reported
88,观察别人的反应决定是否继续训练,NaN,NaN,missing
89,放弃原本想做的力量训练,10.0,31.25,reported
90,离开健身房,10.0,31.25,reported
91,其他 [详情],NaN,NaN,missing


## 新增实验 3：组间比较

比较最常使用自由重量区者与其他人，以及有系统力量训练经验者与其他人。组别非常不平衡，使用 Mann--Whitney $U$，结果只辅助解释。

In [8]:
group_comparisons = pd.concat(
    [
        mann_whitney_group_table(
            participant_scores,
            "free_weight_user",
            corr_variables,
            "最常使用自由重量区",
        ),
        mann_whitney_group_table(
            participant_scores,
            "systematic_strength_training",
            corr_variables,
            "有系统力量训练经验",
        ),
    ],
    ignore_index=True,
)
display(group_comparisons.assign(
    yes_mean=lambda d: d["yes_mean"].round(2),
    yes_sd=lambda d: d["yes_sd"].round(2),
    no_mean=lambda d: d["no_mean"].round(2),
    no_sd=lambda d: d["no_sd"].round(2),
    p=lambda d: d["p"].round(4),
))

,group,variable,yes_n,yes_mean,yes_sd,no_n,no_mean,no_sd,mann_whitney_u,p
0,最常使用自由重量区,空间压迫/训练焦虑,5,2.60,1.05,27,3.92,0.50,16.0,0.0081
1,最常使用自由重量区,社交媒体审美内化,5,2.08,0.68,27,2.77,0.94,34.5,0.0914
2,最常使用自由重量区,训练自我效能,5,3.88,0.76,27,3.42,0.96,86.0,0.3464
3,最常使用自由重量区,干预偏好指数,5,3.77,0.28,27,4.02,0.57,43.5,0.2220
4,最常使用自由重量区,自由重量区回避频率,5,1.40,0.55,27,3.41,1.19,10.0,0.0024
5,有系统力量训练经验,空间压迫/训练焦虑,8,3.18,1.03,24,3.90,0.58,56.5,0.0896
6,有系统力量训练经验,社交媒体审美内化,8,2.03,0.89,24,2.87,0.86,47.5,0.0366
7,有系统力量训练经验,训练自我效能,8,3.82,0.89,24,3.38,0.95,117.5,0.3570
8,有系统力量训练经验,干预偏好指数,8,3.93,0.47,24,3.99,0.57,83.5,0.6011
9,有系统力量训练经验,自由重量区回避频率,8,1.88,0.83,24,3.50,1.22,28.0,0.0026
